# Notebook 3: Failure Case Analysis
### DL Capstone — Hair Change Tracking via U-Net Segmentation
**Ramdeobaba University | Dept. of Electronics Engineering | Semester V**

This notebook explores the 20 worst predictions made by the model on the test set to analyze limitations.

---

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

ROOT = Path(r'C:\Users\Aliasgar\Desktop\DL Capstone\DL_Capstone_HairSegmentation')
WORST_PATH = ROOT / 'outputs' / 'results' / 'worst_predictions.csv'
IMG_DIR = ROOT / 'data' / 'processed' / 'images'
MASK_DIR = ROOT / 'data' / 'processed' / 'masks'


## 1. Load Worst Predictions

In [ ]:
if WORST_PATH.exists():
    df_worst = pd.read_csv(WORST_PATH)
    print("Top 10 Worst Predictions (Lowest Dice Scores):\n")
    display(df_worst.head(10))
else:
    print("Worst predictions file not found! Run evaluate.py first.")


## 2. Visualize Failure Modes

In [ ]:
import torch
import sys
sys.path.insert(0, str(ROOT / 'src'))
from model import UNet
from predict import load_image_tensor, predict_hair_mask

# Load Model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_path = ROOT / 'outputs' / 'models' / 'best_model.pth'
model = UNet(in_channels=3, out_channels=1, features=[32, 64, 128, 256]).to(device)
model.load_state_dict(torch.load(model_path, map_location=device)['model_state_dict'])
model.eval()
print("Model loaded successfully.")


In [ ]:
if WORST_PATH.exists():
    fig, axes = plt.subplots(4, 4, figsize=(15, 15))
    axes = axes.flatten()
    
    for i in range(4):  # Visualize top 4 worst cases
        row = df_worst.iloc[i]
        img_name = row['Image']
        dice = row['Dice']
        
        img_path = str(IMG_DIR / img_name)
        gt_path = str(MASK_DIR / img_name)
        
        # Predict
        tensor, orig_img = load_image_tensor(img_path)
        pred_mask = predict_hair_mask(model, tensor, device)
        pred_mask_np = pred_mask.cpu().numpy()[0, 0] * 255
        
        # Ground truth
        gt_mask = np.array(Image.open(gt_path).convert('L'))
        
        # Overlays
        orig_np = np.array(orig_img)
        
        overlay_gt = orig_np.copy()
        overlay_gt[gt_mask > 128] = (overlay_gt[gt_mask > 128]*0.5 + np.array([0,255,0])*0.5).astype(np.uint8)
        
        overlay_pred = orig_np.copy()
        overlay_pred[pred_mask_np > 128] = (overlay_pred[pred_mask_np > 128]*0.5 + np.array([255,0,0])*0.5).astype(np.uint8)
        
        idx = i * 4
        axes[idx].imshow(orig_img); axes[idx].set_title(f'{img_name}\nDice: {dice:.4f}')
        axes[idx+1].imshow(gt_mask, cmap='gray'); axes[idx+1].set_title('Ground Truth')
        axes[idx+2].imshow(overlay_gt); axes[idx+2].set_title('GT Overlay (Green)')
        axes[idx+3].imshow(overlay_pred); axes[idx+3].set_title('Pred Overlay (Red)')
        
    for ax in axes:
        ax.axis('off')
        
    plt.tight_layout()
    plt.show()


## 3. Categorization of Failures
Based on the worst predictions visualized above, we typically observe the following issues:

1. **Hats/Headwear:** The model struggles to differentiate between a hat and hair if the hat covers the entire head.
2. **Extreme lighting:** Very dark shadows blending with black hair cause under-segmentation.
3. **Complex background:** Similar color/texture in the background causes false positive hair predictions.
4. **Baldness / Sparse hair:** CelebAMask-HQ labels some non-hair regions incorrectly, or the model expects hair where there is none.